In [1]:
import pandas as pd
from Bio import SeqIO

In [2]:
def read_fasta(file):
    with open(file, 'r') as f:
        lines = f.readlines()
    names = []
    seqs = []
    seq = ""
    for line in lines: 
        if line.startswith('>'): 
            names.append(line.replace(">", "").strip())
            if len(names) > 1:
                seqs.append(seq)
                seq = ""
        else:
            seq += line.strip()
    
    seqs.append(seq)

    return pd.DataFrame({'name' : names, 'sequence' : seqs})

In [3]:
full_seqs = read_fasta('full_length_DNA_seqs.txt')
full_seqs['cds'] = ["".join([char for char in seq if char.isupper()]) for seq in full_seqs['sequence']]

In [4]:
def translate(seq):
     
    table = {
        'ATA':'I', 'ATC':'I', 'ATT':'I', 'ATG':'M',
        'ACA':'T', 'ACC':'T', 'ACG':'T', 'ACT':'T',
        'AAC':'N', 'AAT':'N', 'AAA':'K', 'AAG':'K',
        'AGC':'S', 'AGT':'S', 'AGA':'R', 'AGG':'R',                 
        'CTA':'L', 'CTC':'L', 'CTG':'L', 'CTT':'L',
        'CCA':'P', 'CCC':'P', 'CCG':'P', 'CCT':'P',
        'CAC':'H', 'CAT':'H', 'CAA':'Q', 'CAG':'Q',
        'CGA':'R', 'CGC':'R', 'CGG':'R', 'CGT':'R',
        'GTA':'V', 'GTC':'V', 'GTG':'V', 'GTT':'V',
        'GCA':'A', 'GCC':'A', 'GCG':'A', 'GCT':'A',
        'GAC':'D', 'GAT':'D', 'GAA':'E', 'GAG':'E',
        'GGA':'G', 'GGC':'G', 'GGG':'G', 'GGT':'G',
        'TCA':'S', 'TCC':'S', 'TCG':'S', 'TCT':'S',
        'TTC':'F', 'TTT':'F', 'TTA':'L', 'TTG':'L',
        'TAC':'Y', 'TAT':'Y', 'TAA':'_', 'TAG':'_',
        'TGC':'C', 'TGT':'C', 'TGA':'_', 'TGG':'W',
    }
    protein =""
    if len(seq)%3 == 0:
        for i in range(0, len(seq), 3):
            codon = seq[i:i + 3]
            protein+= table[codon]
    return protein

In [5]:
full_seqs['protein_seq'] = full_seqs['cds'].apply(translate)

In [6]:
AD_seqs = pd.read_excel('AD_sequences_short.xlsx', sheet_name='FC>=1.5, P<=1E-7')

/Users/claireleblanc/miniconda3/envs/basic_env/lib/python3.13/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


In [7]:
merged_seqs = AD_seqs.merge(full_seqs, left_on='TF_name', right_on='name')
merged_seqs.head()

,TF_name,start tAD [relative to TF CDS],end tAD [relative to TF CDS],tAD length [aa],tAD sequence [aa],enrichment (log2),P-value,FDR,Pfam,seg,cast (score >=40),fLPS - SINGLE,fLPS - MULTIPLE,fLPS - WHOLE,EMBOSS pepstats charge,name,sequence,cds,protein_seq
0,HLH3B_FBtr0070506,303,918,205,LNGNQADANRSLSSSPRSHSRNGLLTAPASSGSSVGGSGGGGGNGS...,7.01428,0.000000e+00,0.000000e+00,"[HLH,66-116,9.7e-20,2-54]","9-20,29-62,178-191","[29-63, G-rich, score: 100][2-56, S-rich, scor...","[36-63, AA={G}, ct=17, 9.329e-14][11-56, AA={S...","[11-63, AA={GS}, ct=34, 4.416e-17]","[1-205, AA={G}, ct=31, 9.863e-06][1-205, AA={N...",13.0,HLH3B_FBtr0070506,acagatgagatggtcagactaaactggctgacggaatttatgcctc...,ATGGCCTGGCTACCGAGCAGCTCGAACGGAGCGGATAACGCGGACG...,MAWLPSSSNGADNADERSTASSGSSGHSQTNESPRSGHLNGNGSMR...
1,MTF-1_FBtr0113324,495,1032,179,YAAEEEIPSPWIDAGVLVSKPIIPMAPLTDACVALPTEMPSFVNLK...,3.86131,0.000000e+00,0.000000e+00,NaN,119-128,NaN,"[161-179, AA={C}, ct=6, 3.691e-07][8-85, AA={P...",NaN,NaN,-21.5,MTF-1_FBtr0113324,acagatgagatggtcagactaaactggctgacggaatttatgcctc...,ATGCGCACGCACACCCAGGAGCGGCCCTACAAATGTCCGGAAGATG...,MRTHTQERPYKCPEDDCGKAFTASHHLKTHRRTHTGEKPYPCQEDS...
2,sage_FBtr0081982,174,402,76,CPAVSLNYTLTADGAGLLAYAPTHPHSYSPSVMGGYEKEAHNMGLL...,2.94724,2.370700e-223,9.134073e-221,NaN,NaN,NaN,"[22-56, AA={P}, ct=7, 8.537e-04]",NaN,NaN,-1.0,sage_FBtr0081982,acagatgagatggtcagactaaactggctgacggaatttatgcctc...,ATGACGGATCAACTGCTGAGCTCCACGGCCGGCGTACCGGGTGATC...,MTDQLLSSTAGVPGDPPPVLVPVSATTGGPYLSGGSEGEAESSEQP...
3,CG14451_FBtr0078538,582,792,70,SDRLIFRKSSQETHGKSRYLQAEDWRFLFEMEQLVRGIDDGLVRDP...,3.57737,3.273350e-164,1.037120e-161,NaN,NaN,NaN,NaN,NaN,NaN,-3.5,CG14451_FBtr0078538,acagatgagatggtcagactaaactggctgacggaatttatgcctc...,ATGGAGAACAGGAGGTGCACCGTCGTTGCCGTGGGTGACCTTGTGG...,MENRRCTVVAVGDLVDELGFCCEWCGKVYKSLVRYHNHIVCTHKLP...
4,bin_FBtr0076900,1632,1884,84,YAPTASLVAAGYSYATSAGSLDNGLRSISLQQLPGLSSIQHAQAQA...,2.48625,3.889570e-136,1.150594e-133,NaN,"8-19,36-76","[51-76, H-rich, score: 91]","[51-76, AA={H}, ct=13, 1.961e-15][31-49, AA={Q...","[13-80, AA={QS}, ct=23, 8.681e-07]","[1-84, AA={H}, ct=14, 3.715e-09]",7.0,bin_FBtr0076900,acagatgagatggtcagactaaactggctgacggaatttatgcctc...,ATGATTAAAAGTGAAGAAGCCTCCGATCGTTCGGTTATGACCATGG...,MIKSEEASDRSVMTMDQLGGYYHDPRAHPPFSHPHAHSHPHQHTHT...


In [8]:
# Checking that every AD sequence is present in the full length sequences
for index in merged_seqs.index:
    ad_seq = merged_seqs.loc[index, 'tAD sequence [aa]']
    full_seq = merged_seqs.loc[index, 'protein_seq']

    if ad_seq not in full_seq:
        print("HERE")

In [12]:
merged_seqs_filtered = merged_seqs.drop_duplicates(['TF_name', 'protein_seq'])

In [13]:
# Write the full length sequences to a fasta file
with open("arnold_seqs_with_ADs.fasta", 'w+') as f: 
    for i in merged_seqs_filtered.index: 
        f.write(">" + merged_seqs_filtered.loc[i, 'TF_name'] + "\n")
        f.write(merged_seqs_filtered.loc[i, 'protein_seq'] + "\n")

In [13]:
# Write excel file for later 
merged_seqs.to_csv("arnold_seqs_with_short_ADs.csv")

In [14]:
# Write to a file we could run the consensus predictor on

with open("arnold_seqs_with_ADs.tsv", 'w+') as f: 
    for i in merged_seqs_filtered.index: 
        f.write(">" + merged_seqs_filtered.loc[i, 'TF_name'] + " ")
        f.write(merged_seqs_filtered.loc[i, 'protein_seq'] + "\n")

In [9]:
merged_seqs['TF_name'].value_counts()

TF_name
sage_FBtr0081982       2
CG30020_FBtr0088235    2
exex_FBtr0076751       2
dar1_FBtr0113134       2
HLH3B_FBtr0070506      1
CG3726_FBtr0070858     1
CG17186_FBtr0083858    1
slou_FBtr0084123       1
GATAd_FBtr0079993      1
sim_FBtr0082710        1
gem_FBtr0089759        1
toy_FBtr0089185        1
CG16778_FBtr0100126    1
H_FBtr0083916          1
cas_FBtr0078725        1
CG30431_FBtr0085991    1
otp_FBtr0110772        1
srp_FBtr0100595        1
rn_FBtr0300518         1
CG8388_FBtr0087263     1
ac_FBtr0070072         1
lbe_FBtr0084091        1
Kr-h1_FBtr0079195      1
Eip78C_FBtr0306158     1
Doc2_FBtr0076547       1
CG1620_FBtr0300830     1
Cf2_FBtr0304887        1
HLH54F_FBtr0089547     1
MTF-1_FBtr0113324      1
CG14451_FBtr0078538    1
bin_FBtr0076900        1
pdm2_FBtr0080392       1
Hnf4_FBtr0079747       1
Bteb2_FBtr0070735      1
rgr_FBtr0088692        1
E2f_FBtr0084119        1
ey_FBtr0089235         1
CG10321_FBtr0071660    1
Clk_FBtr0076785        1
Hr39_FBtr0081481 